In [ ]:
# 10_cmip6_ml_correction.ipynb
# Apply ML bias correction to CMIP6 future data

%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import warnings
from sklearn.ensemble import GradientBoostingRegressor

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.drought_indices import calculate_spei, extract_drought_events
from src.drought_probability import calculate_return_periods

print("="*60)
print("STEP 1: LOAD HISTORICAL GAUGE METADATA & TRUTH")
print("="*60)

# 1. Load historical metadata to get elev_m and lulc_class
hist_data_path = PROJECT_ROOT / "outputs" / "processed" / "sarawak_ML_corrected.parquet"
hist_gauge = pd.read_parquet(hist_data_path)
hist_gauge['station_id'] = hist_gauge['station_id'].astype(str)

# Create a clean metadata map
meta_map = hist_gauge[['station_id', 'elev_m', 'lulc_class']].drop_duplicates().set_index('station_id')
print(f"✅ Loaded metadata for {len(meta_map)} stations")

print("\n" + "="*60)
print("STEP 2: TRAIN CMIP6-SPECIFIC ML CORRECTORS")
print("="*60)

cmip6_dir = PROJECT_ROOT / "outputs" / "processed" / "cmip6_ensemble"
cmip6_files = list(cmip6_dir.glob("sarawak_cmip6_*.parquet"))
cmip6_files = [f for f in cmip6_files if "master" not in f.name]

models = set([f.stem.split('_')[2] for f in cmip6_files])
print(f"Found {len(models)} CMIP6 models: {models}")

model_correctors = {}

for model in models:
    print(f"\n🔧 Training corrector for {model}...")
    
    hist_files = [f for f in cmip6_files if model in f.stem and 'historical' in f.stem]
    if not hist_files:
        print(f"  ⚠️ No historical data found for {model}. Skipping.")
        continue
        
    df_hist = pd.read_parquet(hist_files[0])
    df_hist['station_id'] = df_hist['station_id'].astype(str)
    
    # CRITICAL FIX: Map elev_m and lulc_class from historical gauge data
    df_hist = df_hist.merge(meta_map, left_on='station_id', right_index=True, how='left')
    df_hist['elev_m'] = df_hist['elev_m'].fillna(0)
    df_hist['lulc_class'] = df_hist['lulc_class'].fillna(0).astype(int)
    df_hist['lulc_code'] = pd.Categorical(df_hist['lulc_class']).codes
    df_hist['month'] = df_hist['date'].dt.month
    
    # Merge with gauge truth
    train_df = df_hist.merge(
        hist_gauge[['station_id', 'date', 'Gauge_Observed']], 
        on=['station_id', 'date'], how='inner'
    ).dropna()
    
    # Use .values to prevent scikit-learn feature name warnings
    X_train = train_df[['P_future', 'elev_m', 'lulc_code', 'month']].values
    y_train = train_df['Gauge_Observed'].values
    
    gbr = GradientBoostingRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)
    gbr.fit(X_train, y_train)
    
    model_correctors[model] = gbr
    print(f"  ✅ Trained on {len(X_train)} historical samples.")

print("\n" + "="*60)
print("STEP 3: APPLY CORRECTION TO FUTURE & EXTRACT DROUGHTS")
print("="*60)

all_corrected_events = []

for cmip6_file in cmip6_files:
    parts = cmip6_file.stem.replace("sarawak_cmip6_", "").split("_")
    model_name = parts[0]
    scenario = parts[1]
    timeframe = parts[2]
    
    if timeframe == 'historical':
        continue
        
    if model_name not in model_correctors:
        continue
        
    print(f"\n🚀 Processing: {model_name} | {scenario} | {timeframe}")
    
    df_future = pd.read_parquet(cmip6_file)
    df_future['station_id'] = df_future['station_id'].astype(str)
    
    # CRITICAL FIX: Map elev_m and lulc_class for future data too
    df_future = df_future.merge(meta_map, left_on='station_id', right_index=True, how='left')
    df_future['elev_m'] = df_future['elev_m'].fillna(0)
    df_future['lulc_class'] = df_future['lulc_class'].fillna(0).astype(int)
    df_future['lulc_code'] = pd.Categorical(df_future['lulc_class']).codes
    df_future['month'] = df_future['date'].dt.month
    
    # Apply ML correction
    X_pred = df_future[['P_future', 'elev_m', 'lulc_code', 'month']].values
    df_future['P_future_corrected'] = model_correctors[model_name].predict(X_pred)
    df_future['P_future_corrected'] = df_future['P_future_corrected'].clip(lower=0)
    
    # Process each station
    for stn_id in df_future['station_id'].unique():
        stn_data = df_future[df_future['station_id'] == stn_id].copy()
        stn_data = stn_data.sort_values('date').set_index('date')
        
        stn_name = stn_data['station_name'].iloc[0] if 'station_name' in stn_data.columns else f"Station_{stn_id}"
        
        if 'P_future_corrected' in stn_data.columns and 'PET_future' in stn_data.columns:
            spei_12 = calculate_spei(stn_data['P_future_corrected'], stn_data['PET_future'], scale=12)
            stn_data['SPEI_12'] = spei_12
            
            events = extract_drought_events(stn_data['SPEI_12'], threshold=-1.0, min_duration=2, inter_event_time=1)
            
            if not events.empty:
                total_months = len(stn_data)
                events = calculate_return_periods(events, total_months)
                
                events['station_id'] = stn_id
                events['station_name'] = stn_name
                events['model'] = model_name
                events['scenario'] = scenario
                events['timeframe'] = timeframe
                
                all_corrected_events.append(events)

# 4. Save to CSV
if all_corrected_events:
    final_events_df = pd.concat(all_corrected_events, ignore_index=True)
    output_csv = PROJECT_ROOT / "outputs" / "processed" / "future_drought_events_corrected.csv"
    final_events_df.to_csv(output_csv, index=False)
    
    print("\n" + "="*60)
    print(f"✅ SUCCESS! Saved {len(final_events_df)} ML-corrected future drought events")
    print(f"   Output: {output_csv}")
    print("="*60)
else:
    print("\n⚠️ No future drought events found.")